In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import os
import yfinance as yf

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping


warnings.simplefilter(action="ignore", category=FutureWarning)

In [3]:
import os
import random

def fijar_semillas(semilla=42):
    # 1. Fijar semilla de Python
    os.environ['PYTHONHASHSEED'] = str(semilla)
    random.seed(semilla)
    
    # 2. Fijar semilla de NumPy
    np.random.seed(semilla)
    
    # 3. Fijar semilla de TensorFlow/Keras
    tf.random.set_seed(semilla)
    
    print(f"[*] Semillas fijadas a {semilla} para asegurar reproducibilidad.")

# Llamar a la función antes de crear ningún modelo ni dividir datos
fijar_semillas(42)

[*] Semillas fijadas a 42 para asegurar reproducibilidad.


In [4]:
# =====================================================================
# 1. DESCARGA Y PREPARACIÓN DE DATOS: PORTFOLIO DOLLAR BARS
# =====================================================================
print("Descargando datos de Yahoo Finance (Precios y Volumen)...")
start_date = '1960-01-01'
tickers_validos = ['AEP', 'BA', 'CAT', 'CNP', 'CVX', 'DIS', 'DTE', 'ED', 'GD', 'GE', 
                   'HON', 'HPQ', 'IBM', 'IP', 'JNJ', 'KO', 'KR', 'MMM', 'MO', 'MRK', 
                   'MSI', 'PG', 'XOM']

# Descargamos todos los datos (OHLCV)
datos_yahoo = yf.download(tickers_validos, start=start_date, auto_adjust=True, progress=False)

# Separamos Precios de Cierre y Volúmenes, eliminando activos con datos faltantes
precios_close = datos_yahoo['Close'].dropna(axis=1)
volumenes = datos_yahoo['Volume'].dropna(axis=1)

print(f"Datos cronológicos originales (Time Bars): {precios_close.shape}")

# ---------------------------------------------------------------------
# A) Cálculo del Dollar Volume del Portfolio
# ---------------------------------------------------------------------
# Multiplicamos precio por volumen para cada activo (Dólares negociados por empresa y día)
dollar_volume_individual = precios_close * volumenes

# Sumamos horizontalmente para obtener los Dólares negociados por TODAS las 23 empresas ese día
portfolio_dollar_volume = dollar_volume_individual.sum(axis=1)


# ---------------------------------------------------------------------
# B) Función de Muestreo de Dollar Bars
# ---------------------------------------------------------------------
def sample_portfolio_dollar_bars(df_precios, serie_dollar_vol, threshold):
    """
    Recorre la serie cronológica acumulando dólares negociados.
    Cuando el acumulado supera el 'threshold' (umbral), guarda la fecha,
    cierra la barra y resetea el contador.
    """
    fechas_barras = []
    vol_acumulado = 0.0
    
    # Extraemos fechas y volúmenes para iterar rápidamente
    fechas = serie_dollar_vol.index
    volumenes_diarios = serie_dollar_vol.values
    
    for fecha, vol in zip(fechas, volumenes_diarios):
        # Ignoramos NaNs que puedan surgir en datos muy antiguos
        if np.isnan(vol):
            continue
            
        vol_acumulado += vol
        
        # ¿Hemos superado la "Triple Barrera" de volumen de dinero?
        if vol_acumulado >= threshold:
            fechas_barras.append(fecha)
            vol_acumulado = 0.0  # Reset de la barra
            
    # Filtramos la matriz original de 23 activos usando solo las fechas donde se cerró una barra
    return df_precios.loc[fechas_barras]

# ---------------------------------------------------------------------
# C) Generación de la nueva matriz indexada por Información
# ---------------------------------------------------------------------
# Definir el umbral (Threshold). 
# Prado recomienda usar una fracción del volumen total, o un múltiplo de la media diaria.
# Por ejemplo: queremos que cada barra contenga la cantidad de dólares que, EN MEDIA, se negocian en 5 días.
umbral_dolares = portfolio_dollar_volume.mean() * 5 

precios_dollar_bars = sample_portfolio_dollar_bars(precios_close, portfolio_dollar_volume, umbral_dolares)

print(f"Nueva matriz tras compresión (Portfolio Dollar Bars): {precios_dollar_bars.shape}")

# (NOTA: Aún no calculamos retornos. Dejaremos precios_dollar_bars intacto para aplicarle FracDiff después)

Descargando datos de Yahoo Finance (Precios y Volumen)...


c:\Users\Joseph\AppData\Local\Programs\Python\Python313\Lib\site-packages\yfinance\scrapers\history.py:144: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end_dt = pd.Timestamp.utcnow().tz_convert(tz)
c:\Users\Joseph\AppData\Local\Programs\Python\Python313\Lib\site-packages\yfinance\scrapers\history.py:201: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
c:\Users\Joseph\AppData\Local\Programs\Python\Python313\Lib\site-packages\yfinance\scrapers\history.py:144: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end_dt = pd.Timestamp.utcnow().tz_convert(tz)
c:\Users\Joseph\AppData\Local\Programs\Python\Python313\Lib\site-packages\yfinance\scrapers\history.py:201: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a f

Datos cronológicos originales (Time Bars): (16196, 23)
Nueva matriz tras compresión (Portfolio Dollar Bars): (2507, 23)


In [5]:
# =====================================================================
# 2. TRANSFORMACIÓN LÓPEZ DE PRADO: DIFERENCIACIÓN FRACCIONARIA (FFD)
# =====================================================================

def obtener_pesos_fracdiff(d, umbral=1e-4):
    """
    Calcula los pesos para la diferenciación fraccionaria.
    Los pesos disminuyen iterativamente. Se cortan cuando el valor
    absoluto del peso es menor que el 'umbral' (threshold).
    """
    w = [1.]
    k = 1
    while True:
        # Fórmula matemática recursiva de López de Prado para los pesos
        w_k = -w[-1] / k * (d - k + 1)
        if abs(w_k) < umbral:
            break
        w.append(w_k)
        k += 1
    
    # Invertimos los pesos para aplicarlos cronológicamente (el más reciente tiene peso 1)
    return np.array(w[::-1])

def aplicar_fracdiff_ffd(df_precios, d, umbral=1e-4):
    """
    Aplica FracDiff a un DataFrame entero usando una ventana fija (FFD).
    Esto evita la pérdida excesiva de datos al inicio de la serie.
    """
    print(f"Calculando Diferenciación Fraccionaria con d={d}...")
    
    # 1. Obtener los pesos matemáticos
    w = obtener_pesos_fracdiff(d, umbral)
    ventana = len(w)
    print(f"Tamaño de la ventana de memoria (días/barras retenidas): {ventana}")
    
    # Transformamos el DF a logaritmos para estabilizar la varianza antes de diferenciar
    df_log = np.log(df_precios)
    
    # 2. Crear un DataFrame vacío para los resultados
    df_diff = pd.DataFrame(index=df_log.index, columns=df_log.columns)
    
    # 3. Aplicar el producto punto (dot product) deslizando la ventana
    # Multiplicamos el vector de pesos (1D) por la matriz de precios de la ventana (2D)
    for i in range(ventana - 1, len(df_log)):
        # Tomamos el bloque temporal exacto de tamaño 'ventana' hasta el día 'i'
        corte = df_log.iloc[i - ventana + 1 : i + 1]
        
        # np.dot suma el peso histórico * precio histórico para los 23 activos a la vez
        df_diff.iloc[i] = np.dot(w, corte.values)
        
    # Eliminamos las primeras filas que no tenían historia suficiente para llenar la ventana
    return df_diff.dropna().astype(float)


# ---------------------------------------------------------------------
# EJECUCIÓN DEL ALGORITMO FRACDIFF
# ---------------------------------------------------------------------
# El parámetro 'd' (grado de diferenciación). 
# En el SP500, un valor entre 0.4 y 0.5 suele ser el punto dulce donde
# la serie se vuelve estacionaria (ADF test < 0.05) pero retiene >70% de memoria.
grado_d = 0.45 

# Reemplazamos tu antiguo: returns = np.log(precios_close).diff().dropna()
# Por la nueva matriz con memoria:
fracdiff_features = aplicar_fracdiff_ffd(precios_dollar_bars, d=grado_d)

print(f"Forma de los datos FracDiff finales: {fracdiff_features.shape}")

Calculando Diferenciación Fraccionaria con d=0.45...
Tamaño de la ventana de memoria (días/barras retenidas): 238
Forma de los datos FracDiff finales: (2270, 23)


In [6]:
from tensorflow.keras.utils import to_categorical

# =====================================================================
# 2. DEFINICIÓN DE ARQUITECTURA, TRIPLE BARRERA Y BASELINES
# =====================================================================

def create_triple_barrier_data(data_features, data_prices, input_window_size, output_window_size, pt_limit, sl_limit):
    """
    Sustituye a create_time_series_data.
    Genera etiquetas de clasificación usando el Método de la Triple Barrera.
    """
    X, y = [], []
    
    # Asegurarnos de que son arrays de numpy
    features_array = data_features.values if isinstance(data_features, pd.DataFrame) else data_features
    # Calculamos el precio promedio del portfolio para evaluar si "el mercado" toca la barrera
    precios_array = data_prices.mean(axis=1).values if isinstance(data_prices, pd.DataFrame) else data_prices

    for i in range(len(features_array) - input_window_size - output_window_size + 1):
        # 1. Ventana de Entrada (Input Features: FracDiff)
        input_sequence = features_array[i : i + input_window_size]
        X.append(input_sequence)
        
        # 2. Trayectoria Futura (Output: Precios reales para la barrera)
        p0 = precios_array[i + input_window_size - 1] # Precio en el momento t_0
        future_prices = precios_array[i + input_window_size : i + input_window_size + output_window_size]
        
        # Calcular los retornos acumulados de la trayectoria (Path)
        path_returns = (future_prices / p0) - 1
        
        # 3. Lógica de la Triple Barrera
        etiqueta = 1  # Por defecto, Clase 1 (Plano / Caduca el tiempo sin tocar barreras)
        
        for r in path_returns:
            if r >= pt_limit:     # Barrera Superior (Take Profit)
                etiqueta = 2      # Clase 2 (Sube)
                break
            elif r <= -sl_limit:  # Barrera Inferior (Stop Loss)
                etiqueta = 0      # Clase 0 (Baja)
                break
                
        y.append(etiqueta)

    X = np.array(X)
    # Convertimos a One-Hot Encoding para la capa Softmax de Keras
    y = to_categorical(y, num_classes=3) 
    
    return X, y


def construir_modelo_prado(config, input_shape):
    """Construye un modelo de CLASIFICACIÓN basado en la configuración dada."""
    model = Sequential()
    model.add(Input(shape=input_shape))
    
    # Capa dinámica (LSTM o GRU)
    CapaRecurrente = config['tipo_capa']
    model.add(CapaRecurrente(config['neuronas'], return_sequences=False))
    
    model.add(Dropout(config['dropout']))
    
    # CAMBIO CRÍTICO: 3 neuronas de salida con activación Softmax para clasificación
    model.add(Dense(3, activation='softmax')) 
    
    optimizador = Adam(learning_rate=config['lr'])
    # CAMBIO CRÍTICO: Nueva función de pérdida y métrica
    model.compile(optimizer=optimizador, loss='categorical_crossentropy', metrics=['accuracy'])
    return model


def calcular_baselines_clasificacion(y_test, y_train):
    """
    Calcula la Precisión (Accuracy) para modelos base simples en clasificación.
    Los 'y' entran en One-Hot, sacamos la clase real con argmax.
    """
    y_test_classes = np.argmax(y_test, axis=1)
    y_train_classes = np.argmax(y_train, axis=1)
    
    # 1. Baseline "Always Buy": El mercado es alcista, siempre predecir Clase 2 (Sube)
    acc_always_buy = np.mean(y_test_classes == 2)
    
    # 2. Baseline "Always Flat": Siempre predecir Clase 1 (Plano)
    acc_always_flat = np.mean(y_test_classes == 1)
    
    # 3. Baseline "Estadístico Histórico" (Equivalente al Buy and Hold de Regresión):
    # Predecir siempre la clase que más apareció durante el periodo de Entrenamiento
    clase_mas_frecuente = np.bincount(y_train_classes).argmax()
    acc_most_frequent = np.mean(y_test_classes == clase_mas_frecuente)
    
    return acc_always_buy, acc_always_flat, acc_most_frequent


# Crear carpeta para guardar gráficas si no existe
os.makedirs('graficas_convergencia_tbm', exist_ok=True)

In [7]:
# =====================================================================
# 3. CONFIGURACIÓN DEL EXPERIMENTO
# =====================================================================

input_windows = [5, 10, 30, 90]
output_windows = [1, 5, 30, 90]

# 1. Lista para ventanas con POCA información (In: 5 y 10)
# Definimos el punto de partida común para clonarlo fácilmente
base_configs = [
    {'tipo_capa': GRU,  'neuronas': 32, 'dropout': 0.0, 'lr': 0.001},
    {'tipo_capa': LSTM, 'neuronas': 64, 'dropout': 0.0, 'lr': 0.001}
]

# Inicializamos las 8 listas independientes. 
# Usamos list() para que sean copias independientes y puedas modificarlas en el futuro
hp_in5_corto  = [
    # Ganador IN 5 OUT 1  |||| IN 5 OUT 5  
    # 1. El Campeón Defensor (Lo mantenemos como control para comparar)
    {'tipo_capa': GRU,  'neuronas': 8, 'dropout': 0.0, 'lr': 0.001},
    
    # 2. El Micro-Cerebro Extremo: Si 8 neuronas se atascan, probamos con 4.
    # Obligamos a la red a ser una simple calculadora de tendencias muy básica.
    {'tipo_capa': GRU,  'neuronas': 4, 'dropout': 0.0, 'lr': 0.001},
    
    # 3. El Cambio de Familia: LSTM diminuta.
    # A veces, la forma en la que la LSTM maneja sus puertas internas filtra 
    # mejor el ruido a cortísimo plazo que la GRU.
    {'tipo_capa': LSTM, 'neuronas': 4, 'dropout': 0.0, 'lr': 0.001},
    
    # 4. El "Contrariano": Subimos neuronas pero metemos Dropout.
    # Le damos 16 neuronas para que intente ver algo más complejo, pero le 
    # apagamos el 10% (Dropout 0.1) para que no pueda memorizar el ruido.
    {'tipo_capa': GRU,  'neuronas': 16, 'dropout': 0.1, 'lr': 0.001}

]
'''
la diferencia entre tu modelo (0.010628) y el Buy & Hold (0.010571) es de 0.000057. 
¡Es un empate técnico absoluto! Básicamente, el modelo se da cuenta de que es imposible predecir el ruido de mañana 
con los 5 días anteriores y hace exactamente lo mismo que el Buy & Hold. Si tras ejecutar esta lista sigues a 0.00005 puntos del Buy & Hold,
debes detenerte y aceptarlo como el resultado definitivo
'''



hp_in5_largo  = [

    # Ganador IN 5 OUT 30 ||| Ganador IN 5 OUT 90 
   
    {'tipo_capa': LSTM, 'neuronas': 128, 'dropout': 0.3, 'lr': 0.0005},
    {'tipo_capa': LSTM, 'neuronas': 64, 'dropout': 0.1, 'lr': 0.0005},
    {'tipo_capa': LSTM, 'neuronas': 32, 'dropout': 0.1, 'lr': 0.0005}


]
'''
Si el Buy & Hold sigue ganando por la mínima, tendremos la evidencia definitiva (y documentada en tus gráficas)
de que el problema no es el algoritmo, sino la estacionariedad de los datos de entrada
'''


hp_in10_corto = [
    
    # Ganador IN 10 OUT 1 
    {'tipo_capa': GRU,  'neuronas': 4, 'dropout': 0.0, 'lr': 0.001},


    # Ganador IN 10 OUT 5
    {'tipo_capa': GRU,  'neuronas': 8, 'dropout': 0.0, 'lr': 0.0005},


    {'tipo_capa': GRU,  'neuronas': 12, 'dropout': 0.0, 'lr': 0.001},
]


hp_in10_largo = [

    # Ganador IN 10 OUT 30 
    {'tipo_capa': LSTM, 'neuronas': 128, 'dropout': 0.25, 'lr': 0.00005},

    # 1. El "Termómetro" Ligero: GRU de 16 neuronas.
    # Mucho más rápida y con menos parámetros que la LSTM pesada. 
    # LR moderado y sin Dropout para que vea los datos sin filtros.
    {'tipo_capa': LSTM,  'neuronas': 32, 'dropout': 0.0, 'lr': 0.0005},
    

    # Ganador IN 10 OUT 90 
    # 2. La LSTM Desatada: 32 neuronas, LR estándar (0.001) y SIN Dropout.
    # Vamos a quitarle todos los frenos. Queremos ver si al dejarla correr 
    # es capaz de encontrar algún patrón, o si directamente hace Overfitting.
    {'tipo_capa': LSTM, 'neuronas': 32, 'dropout': 0.0, 'lr': 0.001},
    
    # 3. El Micro-Cerebro para Largo Plazo: Solo 8 neuronas.
    # Si 10 días solo contienen una única señal de tendencia (alcista/bajista), 
    # 8 neuronas son más que suficientes para capturarla sin confundirse con el ruido.
    {'tipo_capa': LSTM,  'neuronas': 16, 'dropout': 0.0, 'lr': 0.001}
    
]

hp_in30_corto = [
    # Ganador IN 30 OUT 1 
    {'tipo_capa': LSTM, 'neuronas': 16, 'dropout': 0.0, 'lr': 0.0005},

    {'tipo_capa': GRU, 'neuronas': 16, 'dropout': 0.0, 'lr': 0.0005},

    # Ganador IN 30 OUT 5
    {'tipo_capa': GRU,  'neuronas': 4,  'dropout': 0.0, 'lr': 0.001},
    {'tipo_capa': GRU,  'neuronas': 8,  'dropout': 0.0, 'lr': 0.001},
    {'tipo_capa': GRU,  'neuronas': 12,  'dropout': 0.0, 'lr': 0.001}

]


hp_in30_largo = [

    # Ganador IN 30 OUT 30  |||| Ganador IN 30 OUT 90 

    {'tipo_capa': LSTM, 'neuronas': 64, 'dropout': 0.0, 'lr': 0.001},
    {'tipo_capa': LSTM, 'neuronas': 32, 'dropout': 0.0, 'lr': 0.001},
    {'tipo_capa': LSTM, 'neuronas': 16, 'dropout': 0.0, 'lr': 0.001},
    {'tipo_capa': LSTM, 'neuronas': 8, 'dropout': 0.0, 'lr': 0.001}
]
    


hp_in90_corto = [
    # Ganador IN 90 OUT 1 
    {'tipo_capa': LSTM, 'neuronas': 16, 'dropout': 0.0, 'lr': 0.001},
    {'tipo_capa': GRU, 'neuronas': 16, 'dropout': 0.0, 'lr': 0.001},

    # Ganador IN 90 OUT 5
    {'tipo_capa': LSTM, 'neuronas': 8, 'dropout': 0.0, 'lr': 0.001},
    {'tipo_capa': GRU, 'neuronas': 8, 'dropout': 0.0, 'lr': 0.001}
]

hp_in90_largo = [
    # Ganador IN 90 OUT 30 
    {'tipo_capa': GRU,  'neuronas': 32, 'dropout': 0.0, 'lr': 0.0005},
    {'tipo_capa': GRU,  'neuronas': 16, 'dropout': 0.0, 'lr': 0.0005},

    # Ganador IN 90 OUT 90
    {'tipo_capa': LSTM, 'neuronas': 128, 'dropout': 0.2, 'lr': 0.00005},
    {'tipo_capa': LSTM, 'neuronas': 64, 'dropout': 0.1, 'lr': 0.0005},
    {'tipo_capa': LSTM, 'neuronas': 32, 'dropout': 0.1, 'lr': 0.0001},
    {'tipo_capa': GRU, 'neuronas': 8, 'dropout': 0.0, 'lr': 0.0001}
    
]

lista_hiperparametros = []

# Matrices para reportar resultados finales de las Redes Recurrentes
matriz_mae_train_rnn = np.zeros((4, 4))
matriz_mae_val_rnn = np.zeros((4, 4))
matriz_mae_rnn = np.zeros((4, 4)) # Esta es la de Test que ya tenías

matriz_mae_naive = np.zeros((4, 4))
matriz_mae_sma = np.zeros((4, 4))
matriz_mae_bh = np.zeros((4, 4))

# Aumentamos la paciencia a 15 épocas
early_stop = EarlyStopping(
    monitor ='val_loss', 
    patience = 20,               # <--- CAMBIO AQUÍ
    restore_best_weights = True  # IMPORTANTE: Que devuelva los pesos de la mejor época
)


In [8]:
# Inicializa las nuevas matrices de Accuracy antes del bucle
matriz_acc_naive_val = np.zeros((len(input_windows), len(output_windows)))
matriz_acc_most_frequent_val = np.zeros((len(input_windows), len(output_windows)))

matriz_acc_naive_test = np.zeros((len(input_windows), len(output_windows)))
matriz_acc_most_frequent_test = np.zeros((len(input_windows), len(output_windows)))

matriz_acc_train_rnn = np.zeros((4, 4))
matriz_acc_val_rnn = np.zeros((4, 4))
matriz_acc_test_rnn = np.zeros((4, 4))

print("\nIniciando entrenamiento de modelos de Clasificación...")

# Definimos los límites de la Triple Barrera (2% de subida o bajada)
PT_LIMIT = 0.02 
SL_LIMIT = 0.02

for i, in_w in enumerate(input_windows):
    for j, out_w in enumerate(output_windows):

        pt_dinamico = 0.01 + (0.001 * out_w)
        sl_dinamico = 0.01 + (0.001 * out_w)

        print(f"\n=======================================================")
        print(f" Ventana Entrada (Memoria): {in_w} barras | Salida (Horizonte): {out_w} barras")
        print(f"=======================================================")
        
        # 1. Crear datos con FracDiff y Triple Barrera
        # features_fracdiff es tu matriz X con memoria
        # precios_dollar_bars es tu matriz Y base para calcular trayectorias
        X, y = create_triple_barrier_data(
            fracdiff_features, 
            precios_dollar_bars, 
            in_w, 
            out_w, 
            pt_limit=pt_dinamico, 
            sl_limit=sl_dinamico
        )
        
        # 2. Separación CRONOLÓGICA: 70% Train, 20% Validacion, 10% Test
        split_1 = int(len(X) * 0.70)
        split_2 = int(len(X) * 0.90)
        
        X_train, y_train = X[:split_1], y[:split_1]
        X_val, y_val = X[split_1:split_2], y[split_1:split_2]
        X_test, y_test = X[split_2:], y[split_2:]
        
        # =====================================================================
        # 3. Baselines de Clasificación
        # =====================================================================
        # Calcular en Validación
        acc_buy_val, acc_flat_val, acc_freq_val = calcular_baselines_clasificacion(y_val, y_train)
        matriz_acc_naive_val[i, j] = acc_buy_val
        matriz_acc_most_frequent_val[i, j] = acc_freq_val
        
        # Calcular en Test
        acc_buy_test, acc_flat_test, acc_freq_test = calcular_baselines_clasificacion(y_test, y_train)
        matriz_acc_naive_test[i, j] = acc_buy_test
        matriz_acc_most_frequent_test[i, j] = acc_freq_test

        print("--- Baselines (Accuracy) VALIDACIÓN ---")
        print(f"Always Buy: {acc_buy_val:.4f} | Always Flat: {acc_flat_val:.4f} | Most Frequent: {acc_freq_val:.4f}")
        print("--- Baselines (Accuracy) TEST ---")
        print(f"Always Buy: {acc_buy_test:.4f} | Always Flat: {acc_flat_test:.4f} | Most Frequent: {acc_freq_test:.4f}\n")

        # 4. Búsqueda del mejor modelo recurrente
        # En clasificación, queremos MINIMIZAR la pérdida (val_loss) o MAXIMIZAR la precisión (val_accuracy). 
        # Minimizar val_loss es matemáticamente más robusto.
        mejor_val_loss = float('inf')
        mejor_modelo = None
        mejor_historial = None
        mejor_config = None

        # Selección de hiperparámetros (reutilizando tus listas)
        if in_w == 5:
            lista_a_probar = hp_in5_corto if out_w in [1, 5] else hp_in5_largo
            nombre_lista = "In:5 Corto" if out_w in [1, 5] else "In:5 Largo"
        elif in_w == 10:
            lista_a_probar = hp_in10_corto if out_w in [1, 5] else hp_in10_largo
            nombre_lista = "In:10 Corto" if out_w in [1, 5] else "In:10 Largo"
        elif in_w == 30:
            lista_a_probar = hp_in30_corto if out_w in [1, 5] else hp_in30_largo
            nombre_lista = "In:30 Corto" if out_w in [1, 5] else "In:30 Largo"
        elif in_w == 90:
            lista_a_probar = hp_in90_corto if out_w in [1, 5] else hp_in90_largo
            nombre_lista = "In:90 Corto" if out_w in [1, 5] else "In:90 Largo"

        print(f" -> Usando banco de pruebas: [{nombre_lista}]")
        
        for config in lista_a_probar:
            capa_nombre = config['tipo_capa'].__name__
            print(f" -> Entrenando: {capa_nombre}, Neuronas: {config['neuronas']}, LR: {config['lr']}, DropOut: {config['dropout']}")
            
            # Construimos el modelo de CLASIFICACIÓN
            modelo = construir_modelo_prado(config, input_shape=(in_w, 23))
            
            historial = modelo.fit(X_train, y_train, 
                                   validation_data=(X_val, y_val),
                                   epochs=50, 
                                   batch_size=64, 
                                   callbacks=[early_stop], 
                                   verbose=0)
            
            val_loss_actual = min(historial.history['val_loss'])
            
            if val_loss_actual < mejor_val_loss:
                mejor_val_loss = val_loss_actual
                mejor_modelo = modelo
                mejor_historial = historial
                mejor_config = config
        
        print(f"\n[GANADOR] {mejor_config['tipo_capa'].__name__} ({mejor_config['neuronas']} neuronas)")
        
        # 5. Evaluación final del GANADOR en TRAIN, VALIDACIÓN y TEST
        # evaluate ahora devuelve [loss, accuracy]
        loss_train, acc_train_ganador = mejor_modelo.evaluate(X_train, y_train, verbose=0)
        loss_val, acc_val_ganador = mejor_modelo.evaluate(X_val, y_val, verbose=0)
        loss_test, acc_test_ganador = mejor_modelo.evaluate(X_test, y_test, verbose=0)
        
        # Guardar en sus respectivas matrices
        matriz_acc_train_rnn[i, j] = acc_train_ganador
        matriz_acc_val_rnn[i, j] = acc_val_ganador
        matriz_acc_test_rnn[i, j] = acc_test_ganador
        
        print(f"Accuracy del Modelo Ganador en TRAIN:      {acc_train_ganador:.4f}")
        print(f"Accuracy del Modelo Ganador en VALIDACIÓN: {acc_val_ganador:.4f}")
        print(f"Accuracy del Modelo Ganador en TEST:       {acc_test_ganador:.4f}")
        
        # 6. Guardar Gráficas de Convergencia (Pérdida y Precisión)
        import matplotlib.pyplot as plt
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        
        nombre_capa = mejor_config['tipo_capa'].__name__
        n_neuronas = mejor_config['neuronas']
        l_rate = mejor_config['lr']
        d_out = mejor_config['dropout']
        fig.suptitle(f"Clasificación {nombre_capa} | Neuronas: {n_neuronas} | LR: {l_rate} | Drop: {d_out}\n(In:{in_w} - Out:{out_w})")
        
        # Gráfica de Pérdida (Loss)
        ax1.plot(mejor_historial.history['loss'], label='Loss Entrenamiento')
        ax1.plot(mejor_historial.history['val_loss'], label='Loss Validación')
        ax1.set_xlabel('Épocas')
        ax1.set_ylabel('Categorical Crossentropy')
        ax1.legend()
        ax1.grid(True)
        
        # Gráfica de Precisión (Accuracy)
        ax2.plot(mejor_historial.history['accuracy'], label='Accuracy Entrenamiento')
        ax2.plot(mejor_historial.history['val_accuracy'], label='Accuracy Validación')
        ax2.set_xlabel('Épocas')
        ax2.set_ylabel('Accuracy')
        ax2.legend()
        ax2.grid(True)
        
        # Guardar la imagen
        nombre_archivo = f"graficas_convergencia_tbm/conver_in{in_w}_out{out_w}.png"
        plt.savefig(nombre_archivo)
        plt.close()


Iniciando entrenamiento de modelos de Clasificación...

 Ventana Entrada (Memoria): 5 barras | Salida (Horizonte): 1 barras
--- Baselines (Accuracy) VALIDACIÓN ---
Always Buy: 0.2428 | Always Flat: 0.5585 | Most Frequent: 0.5585
--- Baselines (Accuracy) TEST ---
Always Buy: 0.1454 | Always Flat: 0.7753 | Most Frequent: 0.7753

 -> Usando banco de pruebas: [In:5 Corto]
 -> Entrenando: GRU, Neuronas: 8, LR: 0.001, DropOut: 0.0
 -> Entrenando: GRU, Neuronas: 4, LR: 0.001, DropOut: 0.0
 -> Entrenando: LSTM, Neuronas: 4, LR: 0.001, DropOut: 0.0
 -> Entrenando: GRU, Neuronas: 16, LR: 0.001, DropOut: 0.1

[GANADOR] GRU (16 neuronas)
Accuracy del Modelo Ganador en TRAIN:      0.5079
Accuracy del Modelo Ganador en VALIDACIÓN: 0.5585
Accuracy del Modelo Ganador en TEST:       0.7753

 Ventana Entrada (Memoria): 5 barras | Salida (Horizonte): 5 barras
--- Baselines (Accuracy) VALIDACIÓN ---
Always Buy: 0.4491 | Always Flat: 0.1770 | Most Frequent: 0.4491
--- Baselines (Accuracy) TEST ---
Always 

In [12]:
# =====================================================================
# 5. RESULTADOS FINALES EN CLASIFICACIÓN (Tablas para tu GitHub y Presentación)
# =====================================================================

print("\n\n" + "="*60)
print("MATRIZ DE RESULTADOS FINALES EN ENTRENAMIENTO (ACCURACY)")
print("="*60)
df_rnn_train_acc = pd.DataFrame(matriz_acc_train_rnn, 
                            index=[f'In_{w}' for w in input_windows], 
                            columns=[f'Out_{w}' for w in output_windows])
print(df_rnn_train_acc)

print("\n" + "="*60)
print("MATRIZ DE RESULTADOS FINALES EN VALIDACIÓN (ACCURACY)")
print("="*60)
df_rnn_val_acc = pd.DataFrame(matriz_acc_val_rnn, 
                          index=[f'In_{w}' for w in input_windows], 
                          columns=[f'Out_{w}' for w in output_windows])
print(df_rnn_val_acc)

print("\n" + "="*60)
print("MATRIZ DE RESULTADOS FINALES EN TEST (ACCURACY)")
print("="*60)
df_rnn_test_acc = pd.DataFrame(matriz_acc_test_rnn, 
                      index=[f'In_{w}' for w in input_windows], 
                      columns=[f'Out_{w}' for w in output_windows])
print(df_rnn_test_acc)

print("\n" + "="*60)
print("MATRIZ BASELINE 'ALWAYS BUY' (VALIDACIÓN)")
print("="*60)
# Usamos la matriz donde guardamos el accuracy de "Always Buy"
df_always_buy_val = pd.DataFrame(matriz_acc_naive_val, 
                        index=[f'In_{w}' for w in input_windows], 
                        columns=[f'Out_{w}' for w in output_windows])
print(df_always_buy_val)

print("\n" + "="*60)
print("MATRIZ BASELINE 'ALWAYS BUY' (TEST)")
print("="*60)
df_always_buy_test = pd.DataFrame(matriz_acc_naive_test, 
                        index=[f'In_{w}' for w in input_windows], 
                        columns=[f'Out_{w}' for w in output_windows])
print(df_always_buy_test)

print("\n" + "="*60)
print("MATRIZ BASELINE 'MOST FREQUENT' (VALIDACIÓN) - Eq. Buy&Hold")
print("="*60)
df_most_freq_val = pd.DataFrame(matriz_acc_most_frequent_val, 
                      index=[f'In_{w}' for w in input_windows], 
                      columns=[f'Out_{w}' for w in output_windows])
print(df_most_freq_val)

print("\n" + "="*60)
print("MATRIZ BASELINE 'MOST FREQUENT' (TEST) - Eq. Buy&Hold")
print("="*60)
df_most_freq_test = pd.DataFrame(matriz_acc_most_frequent_test, 
                     index=[f'In_{w}' for w in input_windows], 
                     columns=[f'Out_{w}' for w in output_windows])
print(df_most_freq_test)



MATRIZ DE RESULTADOS FINALES EN ENTRENAMIENTO (ACCURACY)
          Out_1     Out_5    Out_30    Out_90
In_5   0.747947  0.451899  0.598848  0.594346
In_10  0.747942  0.440076  0.401539  0.593935
In_30  0.754633  0.440819  0.592233  0.590153
In_90  0.764938  0.427350  0.579508  0.579740

MATRIZ DE RESULTADOS FINALES EN VALIDACIÓN (ACCURACY)
          Out_1     Out_5    Out_30    Out_90
In_5   0.803097  0.384956  0.502242  0.525346
In_10  0.803097  0.384444  0.497758  0.525346
In_30  0.814732  0.385650  0.506787  0.520930
In_90  0.837156  0.382488  0.520930  0.519139

MATRIZ DE RESULTADOS FINALES EN TEST (ACCURACY)
          Out_1     Out_5    Out_30    Out_90
In_5   0.973568  0.367257  0.629464  0.564220
In_10  0.973451  0.367257  0.367713  0.562212
In_30  0.973214  0.370536  0.633484  0.567442
In_90  0.977064  0.366972  0.627907  0.555024

MATRIZ BASELINE 'ALWAYS BUY' (VALIDACIÓN)
          Out_1     Out_5    Out_30    Out_90
In_5   0.099558  0.384956  0.502242  0.525346
In_10  0.099